# Section 3: Robustness, Where You Go Beyond (28 pts)

ECON 5200 Midterm Project · Replication Study

**AI is allowed in this notebook**, as a documented co-pilot under P.R.I.M.E. File every exchange, in full, in [`ai-appendix.md`](../ai-appendix.md) under the exercise it belongs to. If you cannot explain a line, you cannot submit it.

At least four exercises the original did not run. Choose from modern estimators, a specification curve, sensitivity to unobserved confounding, inference (clustering at the level of treatment assignment, a wild cluster bootstrap if clusters are few), bandwidth or instrument strength, and placebo tests, or justify an alternative. For each one: what you did, what you found, and what it implies about the original claim.

In [6]:
# Clone Cell:
# Set directory to cloned github repository for working inside Google Colab.
# Runs Replication Notebook 1 in the background to import
# output/headline_table.csv
# Only uncomment this cell if running the notebook from Google Colab

!git clone https://github.com/Zach-Kaplan44/ECON-5200-Replication-Study /content/ECON-5200-Replication-Study
%cd /content/ECON-5200-Replication-Study/notebooks

from pathlib import Path
if not Path("/content/ECON-5200-Replication-Study/output/headline_table.csv").exists():
    !jupyter nbconvert --to notebook --execute /content/ECON-5200-Replication-Study/notebooks/01_positioning_and_replication.ipynb --output /tmp/01_run.ipynb

fatal: destination path '/content/ECON-5200-Replication-Study' already exists and is not an empty directory.
/content/ECON-5200-Replication-Study/notebooks
[NbConvertApp] Converting notebook /content/ECON-5200-Replication-Study/notebooks/01_positioning_and_replication.ipynb to notebook
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
[NbConvertApp] Writing 35232 bytes to /tmp/01_run.ipynb


## Setup

In [11]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as sm

# The repository root, whether this runs from notebooks/ (Jupyter) or from the root (reproduce.py).
ROOT = Path.cwd() if (Path.cwd() / "download_data.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
(ROOT / "output").mkdir(exist_ok=True)

from download_data import MY_PAPER, PAPERS, fetch

## Data

In [12]:
# Set MY_PAPER once, in download_data.py. fetch() downloads into data/ only if a file is missing.
if MY_PAPER:
    paths = fetch(MY_PAPER)
    print(PAPERS[MY_PAPER]["paper"])
    dwi = pd.read_stata(paths[0])
else:
    print("TODO: set MY_PAPER in download_data.py (castle, card, lee, dw, hansen or abortion).")

  data/hansen_dwi.dta  9.5 MB
Hansen (2015), punishment and deterrence in drunk driving


## The four exercises

In [13]:
# Function Creating And Data Prep Cell
# Rebuilds the variables from the replication section
# Creates a function to re-run my replication regressions with options
# For robustness checks, and checks that the numbers match my replication

CUTOFF = 0.08

# Converts low_score into the same units as BAC
dwi['bac_min'] = dwi['low_score'] / 1000

# Age and year fixed effects, as in Hansen's controls
age_dummy = pd.get_dummies(dwi['aged'], drop_first=True, prefix='Age').astype(int)
year_dummy = pd.get_dummies(dwi['year'], drop_first=True, prefix='Year').astype(int)
dwi[age_dummy.columns] = age_dummy
dwi[year_dummy.columns] = year_dummy

RUNNING = 'bac_min'
H = 0.05
KERNEL = "rectangular"
CONTROLS = ['male', 'white'] + list(age_dummy.columns) + list(year_dummy.columns)

def rd_estimate(
    data,                   # the full dataset; never modified
    cutoff,                 # cutoff in integer thousandths of BAC, e.g. 80 or 150
    h,                      # bandwidth in the same integer units, e.g. 50 or 25
    controls=None,          # list of control column names, or None for no controls
    se="cluster",           # "cluster" (grouped by bac_min) or "HC1"
    rule=None,              # "ge", "gt", or None to use the default rule below
    exclude_years=None,     # optional iterable of years to drop before windowing
    outcome="recidivism",
    score="low_score",
):
    """Local linear RD at `cutoff` with a rectangular kernel of half-width `h`.

    Returns (fitted statsmodels result, jump coefficient as an unrounded float).
    """
    # Default treatment rule: the statute defines aggravated DUI as a BAC
    # strictly above 0.15 (Hansen, Table 1), so strictly above at that cutoff
    # and at-or-above everywhere else.
    if rule is None:
        rule = "gt" if cutoff == 150 else "ge"
    if rule not in ("ge", "gt"):
        raise ValueError("rule must be 'ge', 'gt', or None")

    # Year exclusions apply before the window so the bandwidth isn't affected
    # by which rows the year filter removes.
    sample = data
    if exclude_years is not None:
        # isin() against numbers silently matches nothing if year is stored as
        # strings or categories, which would drop no rows and give a wrong
        # answer with no warning, so check the dtype first.
        if not pd.api.types.is_numeric_dtype(data["year"]):
            raise TypeError(
                f"exclude_years requires a numeric 'year' column, but it has "
                f"dtype {data['year'].dtype}. Convert it first, e.g. "
                f"data['year'] = pd.to_numeric(data['year'])."
            )
        sample = sample[~sample["year"].isin(list(exclude_years))]

    # Window on the integer score, both edges included. Comparing integers is
    # exact; a float comparison on BAC can drop boundary rows, as it does at the
    # 0.15 window, where 0.200 - 0.150 exceeds 0.05 in float64.
    window = sample[np.abs(sample[score] - cutoff) <= h].copy()

    # Running variable in BAC units, then recentered so the constant is the
    # intercept at the cutoff and above_cutoff is the jump.
    window["bac_min"] = window[score] / 1000
    window["bac_centered"] = window["bac_min"] - cutoff / 1000

    # Treatment indicator, also built on the integer score to avoid float error
    # at 0.150, where the >= comparison on bac_min is unreliable.
    if rule == "gt":
        window["above_cutoff"] = (window[score] > cutoff).astype(int)
    else:
        window["above_cutoff"] = (window[score] >= cutoff).astype(int)

    # Interaction term: lets the slope differ on either side of the cutoff.
    window["above_centered"] = window["above_cutoff"] * window["bac_centered"]

    # Rectangular kernel = plain unweighted OLS on the rows inside the window.
    X = window[["above_cutoff", "bac_centered", "above_centered"]]

    if controls:
        # Drop controls that are constant in this sample (e.g. year dummies for
        # years that were excluded, or age dummies absent from a narrow window);
        # they're collinear with the constant and would make X singular.
        keep = [c for c in controls if window[c].nunique(dropna=False) > 1]
        X = pd.concat([X, window[keep]], axis=1)

    X = sm.add_constant(X)
    y = window[outcome]

    # Hansen clusters on the running variable; HC1 offered as a robustness check.
    if se == "cluster":
        fit = sm.OLS(y, X).fit(cov_type="cluster",
                               cov_kwds={"groups": window["bac_min"]})
    elif se == "HC1":
        fit = sm.OLS(y, X).fit(cov_type="HC1")
    else:
        raise ValueError("se must be 'cluster' or 'HC1'")

    return fit, float(fit.params["above_cutoff"])

In [15]:
import statsmodels.api as sm

# Check Cell
# Ensure the function matches the same numbers as my replication
# Expected values come from the saved headline table, not from in-memory
# variables, so this cell runs in a fresh notebook.
table_path = ROOT / "output" / "headline_table.csv"
if not table_path.exists():
    raise FileNotFoundError(
        f"{table_path} not found. Run notebook 01 or `python reproduce.py` "
        f"to generate the headline table before running these checks."
    )

# Row labels were saved as the index, so read them back as the index.
headline = pd.read_csv(table_path, index_col=0)

# Each entry: row label in the table, and the kwargs that should reproduce it.
checks = [
    ("Table 3, Panel A, col 1: BAC >= 0.08, h = 0.05",
     dict(cutoff=80, h=50, controls=CONTROLS)),
    ("Table 3, Panel B, col 1: BAC >= 0.08, h = 0.025",
     dict(cutoff=80, h=25, controls=CONTROLS)),
    ("Appendix Table 1, Panel A, col 1: BAC >= 0.08, h = 0.05, no controls",
     dict(cutoff=80, h=50, controls=None)),
    ("Appendix Table 1, Panel B, col 1: BAC >= 0.08, h = 0.025, no controls",
     dict(cutoff=80, h=25, controls=None)),
    ("Table 4, Panel A, col 1: BAC > 0.15, h = 0.05",
     dict(cutoff=150, h=50, controls=CONTROLS)),
    ("Table 4, Panel B, col 1: BAC > 0.15, h = 0.025",
     dict(cutoff=150, h=25, controls=CONTROLS)),
    ("Appendix Table 2, Panel A, col 1: BAC > 0.15, h = 0.05, no controls",
     dict(cutoff=150, h=50, controls=None)),
    ("Appendix Table 2, Panel B, col 1: BAC > 0.15, h = 0.025, no controls",
     dict(cutoff=150, h=25, controls=None)),
]

# Snapshot the columns so we can confirm the function didn't touch the input.
cols_before = list(dwi.columns)

print(f"{'setting':<40}{'function':>12}{'saved':>12}{'diff':>12}{'N':>9}  match")
all_match = True
for label, kwargs in checks:
    expected = float(headline.loc[label, "my_estimate"])  # saved unrounded
    fit, jump = rd_estimate(dwi, **kwargs)                # defaults: cluster SEs, default rule
    diff = jump - expected
    match = np.isclose(jump, expected, rtol=0, atol=1e-10)
    all_match &= match
    short = label.split(": ")[1]                          # trim the table reference for width
    print(f"{short:<40}{jump:>12.6f}{expected:>12.6f}{diff:>12.2e}"
          f"{int(fit.nobs):>9}  {'yes' if match else 'NO'}")

cols_after = list(dwi.columns)
print(f"\nall eight reproduce: {all_match}")
print(f"dwi columns unchanged: {cols_before == cols_after}")

setting                                     function       saved        diff        N  match
BAC >= 0.08, h = 0.05                      -0.022145   -0.022145   -7.98e-17    93899  yes
BAC >= 0.08, h = 0.025                     -0.019260   -0.019260   -6.59e-17    50016  yes
BAC >= 0.08, h = 0.05, no controls         -0.021845   -0.021845   -3.12e-17    93899  yes
BAC >= 0.08, h = 0.025, no controls        -0.019402   -0.019402   -4.51e-17    50016  yes
BAC > 0.15, h = 0.05                       -0.009019   -0.009019   -5.72e-17   139033  yes
BAC > 0.15, h = 0.025                      -0.009922   -0.009922   -2.08e-17    75909  yes
BAC > 0.15, h = 0.05, no controls          -0.009279   -0.009279   -3.82e-17   139033  yes
BAC > 0.15, h = 0.025, no controls         -0.009693   -0.009693   -4.16e-17    75909  yes

all eight reproduce: True
dwi columns unchanged: True


### Exercise 1: Heteroskeadsticity-robust (HC1) Standard Errors

**What I did.** *TODO*

In [18]:
fit_hc1, jump_hc1 = rd_estimate(dwi, cutoff=80, h=50, controls=CONTROLS, se="HC1")
print(f"{jump_hc1:.4f} (HC1 SE {fit_hc1.bse['above_cutoff']:.4f})")

# Fresh names throughout so nothing from the earlier cells is overwritten.
inference_rows = []          # one dict per specification, assembled into the table below
coef_mismatches = []         # labels where the two coefficients aren't bit-identical

for spec_label, spec_kwargs in checks:
    # Same specification, fit twice; `se` is passed separately so the dicts
    # inside `checks` are never mutated.
    fit_cl, jump_cl = rd_estimate(dwi, se="cluster", **spec_kwargs)
    fit_rb, jump_rb = rd_estimate(dwi, se="HC1", **spec_kwargs)

    # Only the covariance estimator differs, so the point estimates come from
    # the same OLS solve and should be exactly equal, not merely close.
    if jump_cl != jump_rb:
        coef_mismatches.append(spec_label)

    # Pull the SE and p-value for the jump coefficient out of each fit.
    se_cluster = float(fit_cl.bse["above_cutoff"])
    se_hc1 = float(fit_rb.bse["above_cutoff"])

    # conf_int() returns a frame indexed by parameter with columns 0 (lower)
    # and 1 (upper); default alpha=0.05 gives the 95% interval. Each fit builds
    # its interval from its own covariance estimator, so the two differ in width
    # but are centered on the same coefficient.
    ci_cl = fit_cl.conf_int().loc["above_cutoff"]
    ci_rb = fit_rb.conf_int().loc["above_cutoff"]

    inference_rows.append({
        "label": spec_label,
        "coef": jump_cl,                       # identical under both, checked above
        "se_cluster": se_cluster,
        "se_hc1": se_hc1,
        "se_ratio": se_hc1 / se_cluster,       # >1 means clustering tightens the SE
        "p_cluster": float(fit_cl.pvalues["above_cutoff"]),
        "p_hc1": float(fit_rb.pvalues["above_cutoff"]),
        "ci_low_cluster": float(ci_cl[0]),
        "ci_high_cluster": float(ci_cl[1]),
        "ci_low_hc1": float(ci_rb[0]),
        "ci_high_hc1": float(ci_rb[1]),
        "N": int(fit_cl.nobs),                 # window is the same, so N matches too
    })

# Row labels as the index, matching how the headline table is saved.
inference_df = pd.DataFrame(inference_rows).set_index("label")

# Report the coefficient check before anything else, since a mismatch would
# mean the two runs weren't actually the same specification.
if coef_mismatches:
    raise AssertionError(
        "Coefficient changed with the SE type in: " + "; ".join(coef_mismatches)
    )
print("coefficients identical under cluster and HC1: True\n")

# Write the table out alongside the other saved output.
inference_path = ROOT / "output" / "robustness_inference.csv"
inference_path.parent.mkdir(parents=True, exist_ok=True)   # no-op if output/ exists
inference_df.to_csv(inference_path)                        # full precision on disk
print(f"saved to {inference_path}\n")

# Print rounded to four decimals; N stays an integer because float_format
# only applies to float columns.
print(inference_df.to_string(float_format=lambda v: f"{v:.4f}"))

-0.0221 (HC1 SE 0.0043)
coefficients identical under cluster and HC1: True

saved to /content/ECON-5200-Replication-Study/output/robustness_inference.csv

                                                                         coef  se_cluster  se_hc1  se_ratio  p_cluster  p_hc1  ci_low_cluster  ci_high_cluster  ci_low_hc1  ci_high_hc1       N
label                                                                                                                                                                                          
Table 3, Panel A, col 1: BAC >= 0.08, h = 0.05                        -0.0221      0.0041  0.0043    1.0542     0.0000 0.0000         -0.0302          -0.0141     -0.0306      -0.0137   93899
Table 3, Panel B, col 1: BAC >= 0.08, h = 0.025                       -0.0193      0.0048  0.0057    1.1844     0.0001 0.0007         -0.0287          -0.0099     -0.0304      -0.0081   50016
Appendix Table 1, Panel A, col 1: BAC >= 0.08, h = 0.05, no controls  -0.0218

**What I found.** *TODO*

**What it implies about the original claim.** *TODO: this part is what is graded.*

### Exercise 2: Placebo test on Recidivism
**What I did.** *TODO*

In [20]:
fit_p50, jump_p50 = rd_estimate(dwi, cutoff=50, h=25, controls=CONTROLS)
print(f"{jump_p50:.4f} (SE: {fit_p50.bse['above_cutoff']:.4f})")

# All new names so nothing from the earlier cells (including fit_p50) is touched.
PLACEBO_H = 25                       # same bandwidth for every row, in thousandths
REAL_CUTOFFS = (80, 150)             # the two statutory thresholds
placebo_cutoffs = (40, 50, 110, 115, 120, 180)
placebo_rows = []                    # one dict per cutoff, assembled into the table below

# Run the real cutoffs alongside the placebos so the table is self-contained;
# sorted so the rows read left to right along the BAC scale.
placebo_specs = sorted(
    [(c, "placebo") for c in placebo_cutoffs] + [(c, "real") for c in REAL_CUTOFFS]
)

for cut, kind in placebo_specs:
    # A placebo window that straddles a real threshold would pick up the actual
    # treatment effect leaking in, so the row would be meaningless. Check the
    # window bounds before fitting, and only for the placebos.
    if kind == "placebo":
        contaminating = [r for r in REAL_CUTOFFS if abs(r - cut) <= PLACEBO_H]
        if contaminating:
            raise ValueError(
                f"Placebo cutoff {cut} with h={PLACEBO_H} spans the window "
                f"[{cut - PLACEBO_H}, {cut + PLACEBO_H}], which contains the real "
                f"cutoff(s) {contaminating}. Narrow the bandwidth or drop this cutoff."
            )

    # Identical specification throughout: h=25, CONTROLS, clustered SEs. The
    # treatment rule falls out of the function's default, so the placebos use
    # >= and the 0.15 row uses >, as in the headline table.
    fit_pl, jump_pl = rd_estimate(dwi, cutoff=cut, h=PLACEBO_H, controls=CONTROLS)

    # 95% interval for the jump, built from this fit's clustered covariance.
    ci_pl = fit_pl.conf_int().loc["above_cutoff"]

    placebo_rows.append({
        "cutoff_bac": cut / 1000,                          # report in BAC units
        "kind": kind,                                      # "real" or "placebo"
        "coef": jump_pl,
        "se": float(fit_pl.bse["above_cutoff"]),
        "p_value": float(fit_pl.pvalues["above_cutoff"]),
        "ci_low": float(ci_pl[0]),
        "ci_high": float(ci_pl[1]),
        "N": int(fit_pl.nobs),
    })

# Index on the cutoff so rows are identified by the thing that varies.
placebo_df = pd.DataFrame(placebo_rows).set_index("cutoff_bac")

# Save unrounded, matching how the other output tables are written.
placebo_path = ROOT / "output" / "robustness_placebo.csv"
placebo_path.parent.mkdir(parents=True, exist_ok=True)     # no-op if output/ exists
placebo_df.to_csv(placebo_path)
print(f"saved to {placebo_path}\n")

# Display only: four decimals for the estimate columns, three for p-values.
# Per-column formatters, since to_string's float_format is applied globally.
placebo_formatters = {
    "coef":    lambda v: f"{v:.4f}",
    "se":      lambda v: f"{v:.4f}",
    "ci_low":  lambda v: f"{v:.4f}",
    "ci_high": lambda v: f"{v:.4f}",
    "p_value": lambda v: f"{v:.3f}",
}
print(placebo_df.to_string(formatters=placebo_formatters))

-0.0184 (SE: 0.0137)
saved to /content/ECON-5200-Replication-Study/output/robustness_placebo.csv

               kind    coef     se p_value  ci_low ci_high      N
cutoff_bac                                                       
0.040       placebo  0.0214 0.0142   0.131 -0.0063  0.0491   9727
0.050       placebo -0.0184 0.0137   0.178 -0.0452  0.0084  16417
0.080          real -0.0193 0.0048   0.000 -0.0287 -0.0099  50016
0.110       placebo  0.0013 0.0043   0.763 -0.0072  0.0098  76985
0.115       placebo -0.0011 0.0036   0.759 -0.0083  0.0060  79001
0.120       placebo  0.0017 0.0035   0.622 -0.0052  0.0086  80194
0.150          real -0.0099 0.0046   0.030 -0.0189 -0.0010  75909
0.180       placebo -0.0026 0.0050   0.595 -0.0124  0.0071  55597


**What I found.** *TODO*

**What it implies about the original claim.** *TODO: this part is what is graded.*

### Exercise 3: TODO name it

**What I did.** *TODO*

**What I found.** *TODO*

**What it implies about the original claim.** *TODO: this part is what is graded.*

### Exercise 4: TODO name it

**What I did.** *TODO*

**What I found.** *TODO*

**What it implies about the original claim.** *TODO: this part is what is graded.*